### Seams
##### Objective 
Learning about Seams

$\text{Seam Carving Algorithm} := \text{Changing Size  <=>  Changing Aspect ratio  <=>  Retargeting and Image}$

$\text{Aspect Ratio} := \text{The ratio between the Width and the Height}$

The main problem in reshaping/resizing an image non-uniformly is the creating of artifact/aliasing. 

The seam Carving algorithm devises a method to increase/reduce the size of an image whilst minimizing the artifact in the image in the process.

$\text{Vertical Seam} := \text{A  path  of  connected  pixels from the top of the image to the bottom, where in each row contains 1 pixel}$

$\text{Horizontal  Seam} := \text{A  path  of  connected  pixels from the Left of the image to the Right, where in each row contains 1 pixel}$

- Pixels in a scene can be connected by either sharing an edge of a pixel or a corner of a pixel

$\text{Seam Carving} := \text{The process of removing one seam from the image}$

- Note: if we remove a veritcal seam we will reduce the **image width** by one pixel.
- Note: if we remove a horizontal seam we will reduce the **image height** by one pixel.

- Seam carving is a local operation, meaning only the area (neighborhood) around the seam will be affected by its removal but the rest of the image won't be affected.

<img src="image_U1/Screenshot 2025-05-26 at 13.25.29.png" height="300" width="280"/> 

<img src="image_U1/Screenshot 2025-05-26 at 13.25.53.png" height="300" width="280"/> 

<img src="image_U1/Screenshot 2025-05-26 at 13.28.32.png" height="300" width="280"/>

<img src="image_U1/Screenshot 2025-05-26 at 13.28.18.png" height="300" width="280"/> 



#### Choosing the seams to remove
$\text{Content Aware Resizing} := \text{removing unoticable seams, so that viewer won't notice}$

Reminder:

$\text{Edge} := \text{Values between two neighboring pixels are relatively large to the whole image}$

This means we look at the magnitude of the gradient for each pixel:
$$ | \Delta I(x,y)| = \sqrt{\left ( \frac{d}{dx}I(x,y)\right)^2 + \left( \frac{d}{dy}I(x,y)\right)^2}$$

$$|| \nabla I(x,y)||_1 = |dx|+|dy|$$

Looking at the gradient magnitude for each pixel, we can create an importance map (meaning deciding which pixel is important to keep and which are less to remove). As such pixels that belong to edges or boundaries have high gradient magnitude and will be visible to the viewer as such **They are important** pixels.

$\text{Conclusion}$

$\text{We must choose a path/seam that contains the minimal overall gradient magnitude}$


$\text{Let S be a Seam, we define}$

$E(S) = \sum_{(x,y) \in S}E(I(x,y))$ 

$\text{where} \  E(I(x,y)) = | \frac{\partial}{\partial x}I(x,y)| + |\frac{\partial}{\partial y}I(x,y)|$

Remember: These are discrete values in our case as we're delaing with pixels.

We want $S^* = argmin_S E(S)$

### Finding the Optimal Seam

given an image of n Columns and m Rows the number of possible seams that can be chosen is computed as follows: 
1. first row we have n options
2. By the definition of a seam and how they are connect, we have 3 options 
3. This is applied for m-1 rows
4. Concluding we have $n3^{(m-1)}$ options to choose from, or if we're doing horizontal $m3^{(n-1)}$.

The option of Brute Forcing our optimal seam isn't possible.

#### Dynamic Programming

1. $\text{Create a matrix M of the exact same dimension as the image}$. 
2. $\text{Compute the gradient magnitude of each pixel in the image and store in the matrix.}$
 $$(M(i,j) = E(i,j)) $$

<p align="center">
    <img src="image_U1/Screenshot 2025-05-26 at 14.02.44.png" height="300" width="300"/> 
</p>



3. $\text{Apply the following formula: }$
$$
M(i,j) = E(i,j) + \min\left\{ M(i-1,j-1),\ M(i-1, j),\ M(i-1, j+1) \right\}
$$
$\text{For each pixel in a row and for each row. Note that on the first row the values are the absolute gradient cost}$

<p align="center">
<img src="image_U1/Screenshot 2025-05-26 at 14.02.12.png" height="300" width="300"/> 

<img src="image_U1/Screenshot 2025-05-26 at 14.03.18.png" height="300" width="300"/> 

<img src="image_U1/Screenshot 2025-05-26 at 14.03.41.png" height="300" width="300"/> 
</p>

4. $\text{Once we reached the last entry in the matrix (bottom right corner)} \\ \text{starting at the last row, we find the minimal entry, and go up to the top choosing the minimal of the 3 pixels above at each row} \\ \text{Or We find the minimal entry on the last column and go left choosing the minimal of the 3 adjacent pixels of the left adjacent column.}$

<p align="center">

<img src="image_U1/Screenshot 2025-05-26 at 14.04.54.png" height="300" width="300"/> 

<img src="image_U1/Screenshot 2025-05-26 at 14.08.35.png" height="300" width="300"/> 

<img src="image_U1/Screenshot 2025-05-26 at 14.08.48.png" height="300" width="300"/> 
</p>

$\text{Complexity} = O(nm)$ Linear on the number of pixels, as we just have to calculate the gradients, fill the matrix and backtrack.



### Removing Multiple seams (k-Seams-Carving)

To remove k seams, the seam carving algorithm is run **k times**, each time on the resulting image from the previous iteration.

#### Naïve Algorithm 

At each iteration i, we:
1.	Compute the energy map of the current image (which reflects the importance of pixels).
2.	Find and remove a single seam with the lowest cumulative energy.
3.	Update the image by removing the selected seam.

#### Inefficient: Recomputing the full energy map from scratch every time is inefficient. 

#### Optimization
**We only need to update the energy values (i.e., gradient costs) of pixels that are directly affected by the seam removal.**

After removing a seam at iteration i, only the pixels adjacent to the removed seam in the updated image are affected. Specifically, you should:

Recalculate the gradient for:

•	The pixels in the same row as the removed seam pixel.

•	And the immediate left and right neighbors of that seam pixel (since their gradients now depend on a different neighborhood after removal).


These are the only pixels whose energy values may have changed due to the seam removal — the rest of the image remains unchanged in structure and content.

$\text{Consider the following seam being removed at iteration i:}$

<img src="image_U1/Screenshot 2025-05-26 at 14.08.48.png" height="300" width="300"/> 

$\text{The row entries that'll need to updated are:}$

[ 5,  8, 12,  3 ]

[14,  7,  6, 12]

[14,  9, 10,  8]

[14, 13, 15, 16]

Then 

[ 5,  8, 12 ]      (removed col 3)

[14,  7, 12 ]      (removed col 2)

[14, 10,  8 ]      (removed col 1)

[14, 15, 16 ]      (removed col 1)


Row 1: cols 1, 2 (→ update M[1][1], M[1][2])

Row 2: cols 0, 1 (→ update M[2][0], M[2][1])

Row 3: cols 0, 1 (→ update M[3][0], M[3][1])



- Row 0 : 2
- Row 1: 1,2
- Row 2: 0,1
- Row 3: 0,1